## Project 1 Implementation. 

Below I have copied the original code from https://deepxde.readthedocs.io/en/latest/demos/pinn_forward/Kovasznay.flow.html while also adding comments to walk through the code. I mark locations where I created new code clearly. 
They created their own python package deepxde. As I chose not to re-implement this code in python, I draw from functionality that they already created occasionally. 

# Modification 1: Implementing a hard constraint. 

In [ ]:
import deepxde as dde
import numpy as np
from deepxde import backend as bkd
import time


# Original flow parameters and domain
Re = 20     #initialize reynolds number
nu = 1 / Re
l = 1 / (2 * nu) - np.sqrt(1 / (4 * nu**2) + 4 * np.pi**2)      #Define lambda

x_min, x_max = -0.5, 1.0        #Set domain bounds
y_min, y_max = -0.5, 1.5


# Original Navier–Stokes residuals
def pde(x, u):
    u_vel, v_vel = u[:, 0:1], u[:, 1:2]     #Extract velocity components

    u_vel_x = dde.grad.jacobian(u, x, i=0, j=0)     #Calculate du/dx
    u_vel_y = dde.grad.jacobian(u, x, i=0, j=1)     #Calculate du/dy
    u_vel_xx = dde.grad.hessian(u, x, component=0, i=0, j=0)    #Calculate d2u/dx2
    u_vel_yy = dde.grad.hessian(u, x, component=0, i=1, j=1)    #Calculate d2u/dy2

    v_vel_x = dde.grad.jacobian(u, x, i=1, j=0)     #Same as above, just for v
    v_vel_y = dde.grad.jacobian(u, x, i=1, j=1)
    v_vel_xx = dde.grad.hessian(u, x, component=1, i=0, j=0)
    v_vel_yy = dde.grad.hessian(u, x, component=1, i=1, j=1)

    p_x = dde.grad.jacobian(u, x, i=2, j=0)         #Calculate pressure gradients
    p_y = dde.grad.jacobian(u, x, i=2, j=1)

    momentum_x = (u_vel * u_vel_x + v_vel * u_vel_y + p_x - nu * (u_vel_xx + u_vel_yy)) #Formulate x momentum eq
    momentum_y = (u_vel * v_vel_x + v_vel * v_vel_y + p_y - nu * (v_vel_xx + v_vel_yy)) #formulate y momentum eq
    continuity = u_vel_x + v_vel_y                                                      #Formulate continuity eq

    return [momentum_x, momentum_y, continuity]


# Exact solution, used for evaluation
def u_func(x):
    return 1 - np.exp(l * x[:, 0:1]) * np.cos(2 * np.pi * x[:, 1:2])    #Exact solution for u, v and p

def v_func(x):
    return (l / (2 * np.pi) * np.exp(l * x[:, 0:1]) * np.sin(2 * np.pi * x[:, 1:2]))

def p_func(x):
    return 0.5 * (1 - np.exp(2 * l * x[:, 0:1]))


# NEW: hard boundary transformation. Justification for G and D in report
def hard_boundary_transform(xy, raw_output):
    x, y = xy[:, 0:1], xy[:, 1:2] #Extract coordinates

    s = (x - x_min) / (x_max - x_min) # Linear interpolation 
    A = (1 - s) * np.exp(l * x_min) + s * np.exp(l * x_max)

    # Fixed fields that match velocity
    G_u = 1 + bkd.exp(l * x) - (1 + bkd.cos(2 * np.pi * y)) * A
    G_v = l / (2 * np.pi) * A * bkd.sin(2 * np.pi * y)

    # Network corrections vanish on every edge
    D = (x - x_min) * (x_max - x) * (y - y_min) * (y_max - y)

    u = G_u + D * raw_output[:, 0:1] #Get u and v
    v = G_v + D * raw_output[:, 1:2]

    # Pressure condition only at outlet.
    p_outlet = 0.5 * (1 - np.exp(2 * l * x_max))
    p = p_outlet + (x_max - x) * raw_output[:, 2:3]

    return bkd.concat([u, v, p], axis=1)


spatial_domain = dde.geometry.Rectangle(xmin=[x_min, y_min], xmax=[x_max, y_max]) #Defines domain to train on

# Shared evaluation points for both models
dde.config.set_random_seed(1234)
X = spatial_domain.random_points(100000)


def train_and_evaluate(hard_bc=False, seed=0):
    name = "Hard BC" if hard_bc else "Soft BC"
    print(f"\nStarting {name}, Re = {Re}")

    # Reset TensorFlow state before creating each fresh model
    if dde.backend.backend_name == "tensorflow.compat.v1":
        bkd.tf.reset_default_graph()
    elif dde.backend.backend_name == "tensorflow":
        bkd.tf.keras.backend.clear_session()

    # Use the same seed for sampling and network initialization
    dde.config.set_random_seed(seed)

    if dde.backend.backend_name == "tensorflow":
        bkd.tf.keras.utils.set_random_seed(seed)

    if hard_bc:
        # CHANGED: no boundary penalties
        boundary_conditions = [] #No longer boundary penalties w/ hard constraint

    else:
        # Original soft boundary conditions from the documentation
        boundary_u = dde.icbc.DirichletBC(
            spatial_domain,
            u_func,
            lambda _, on_boundary: on_boundary,
            component=0,
        )

        boundary_v = dde.icbc.DirichletBC(
            spatial_domain,
            v_func,
            lambda _, on_boundary: on_boundary,
            component=1,
        )

        boundary_p = dde.icbc.DirichletBC(
            spatial_domain,
            p_func,
            lambda x, on_boundary: (
                on_boundary and dde.utils.isclose(x[0], x_max)
            ),
            component=2,
        )

        boundary_conditions = [boundary_u, boundary_v, boundary_p]

    data = dde.data.PDE(
        spatial_domain,
        pde,
        boundary_conditions,
        num_domain=2601,
        num_boundary=400,  # These points still contribute to the PDE residual
        num_test=100000,
    )

    net = dde.nn.FNN([2] + 4 * [50] + [3], "tanh", "Glorot normal")

    if hard_bc:
        # NEW: apply hard constraint
        net.apply_output_transform(hard_boundary_transform)

    model = dde.Model(data, net) #Initialize model

    # Training
    start = time.perf_counter()

    model.compile("adam", lr=1e-3)
    model.train(iterations=30000)

    model.compile("L-BFGS") #So the original formulation switches to a L-BFGS optimizer after 30000 iterations. It shuts off on it's own. Didn't look into this deeply but it's from the original implementation. 
    losshistory, train_state = model.train()

    training_time = time.perf_counter() - start

    # Original evaluation
    output = model.predict(X)

    #Output predictions
    u_pred = output[:, 0]
    v_pred = output[:, 1]
    p_pred = output[:, 2]

    #Grab analytical solution
    u_exact = u_func(X).reshape(-1)
    v_exact = v_func(X).reshape(-1)
    p_exact = p_func(X).reshape(-1)

    f = model.predict(X, operator=pde) #Evaluate residuals from pde

    l2_difference_u = dde.metrics.l2_relative_error(u_exact, u_pred) #Calculates error against exact solution. 
    l2_difference_v = dde.metrics.l2_relative_error(v_exact, v_pred)
    l2_difference_p = dde.metrics.l2_relative_error(p_exact, p_pred)

    residual = np.mean(np.absolute(f))

    print(f"\n{name} results")
    print("Mean residual:", residual)
    print("L2 relative error in u:", l2_difference_u)
    print("L2 relative error in v:", l2_difference_v)
    print("L2 relative error in p:", l2_difference_p)
    print("Training time (s):", training_time)

    return {
        "model": model,
        "losshistory": losshistory,
        "train_state": train_state,
        "u_error": l2_difference_u,
        "v_error": l2_difference_v,
        "p_error": l2_difference_p,
        "residual": residual,
        "training_time": training_time,
    }


# Train the original soft model first, then a fresh hard model
results = {}
results["Soft BC"] = train_and_evaluate(hard_bc=False, seed=0)
results["Hard BC"] = train_and_evaluate(hard_bc=True, seed=0)


# Compare both models after training
print("\nComparison")
print(
    f"{'Model':<12} {'u error':>12} {'v error':>12} "
    f"{'p error':>12} {'Residual':>12} {'Time (s)':>12}"
)

for name, result in results.items():
    print(
        f"{name:<12} "
        f"{result['u_error']:>12.3e} "
        f"{result['v_error']:>12.3e} "
        f"{result['p_error']:>12.3e} "
        f"{result['residual']:>12.3e} "
        f"{result['training_time']:>12.1f}"
    )